<div>
<center><img src="../assets/Flux-logo.svg" width="360"/></center>
</div>

<div style="background:linear-gradient(90deg,#036291 0%,#91C2D8 100%);padding:20px 26px;border-radius:10px;border-left:10px solid #D9A441;margin-top:18px">
<h1 style="margin:0;color:#ffffff">Module 2: The Flux Operator, Inside Usernetes</h1>
<p style="margin:6px 0 0 0;color:#DCECF4;font-size:15px">Turducken: Flux in Kubernetes, in a Flux job</p>
<p style="margin:2px 0 0 0;color:#DCECF4;font-size:13px">SC26 &middot; Chicago &middot; November 2026</p>
</div>

The cluster from the last notebook is running. Now we put Flux back inside it.

First LAMMPS the ordinary way, straight through the Flux instance the notebook is already
running in. Then the same LAMMPS run as a MiniCluster, scheduled by a Flux instance that
the Flux Operator brings up inside Usernetes, which is itself a Flux job.


## 2. Run LAMMPS with Flux

<div class="alert alert-block" style="background-color:#91C2D8;color:#06293D">
<span style="font-weight:600">Description:</span> Running LAMMPS with Flux in two ways 🍳 -- directly on the virtual machine, analogous to "bare metal" and then in user-space Kubernetes.
</div>

LAMMPS -- the Large-scale Atomic/Molecular Massively Parallel Simulator -- is a widely used molecular dynamics (MD) open-source code. You can learn more about it [here](https://www.lammps.org). Let's start with a simple LAMMPS run, specifically LAMMPS with Flux Framework. This would be akin to sitting in an allocation you've created and interacting with your workload manager.  First, inspect the resources that you have.

```bash
flux resource list
```

Run a LAMMPS job that uses them!

```bash
flux run -N1 -n 64 -o cpu-affinity=per-task --cwd /opt/lammps/examples/reaxff/HNS lmp -v x 4 -v y 4 -v z 4 -in in.reaxff.hns -nocite
```

Just as we learned in the Module 1, if you change `flux run` to `flux submit` the job will be non-blocking. You can also change the LAMMPS problem size to be larger to have a longer running time. 

### Flux in Kubernetes
Next, we are going to make a [turducken](https://en.wikipedia.org/wiki/Turducken) - running Flux in Kubernetes, which (in our HPC setups) is already running under Flux! 
If we had more than one physical node, this would give us a powerful means to run HPC workloads in Kubernetes, with features that HPC cannot easily support such as elasticity and dynamism, declarative management, and modularity. 

<table>
    <tr>
<td style="width: 250px;"><img src="img/flux-usernetes-turkducken.png"></td>
        <td>Flux Framework and Usernetes setup. Your virtual machine is provisioned with both the workload manager Flux
Framework and User-space Kubernetes (1). We are emulating in this notebook you, the HPC user, running a batch job with Flux that has brought up your own user-space Kubernetes cluster (2). We will then deploy the Flux Operator (3) inside that cluster to run LAMMPS.</td>
    </tr>
</table>

Let's install the Flux Operator. Note we are installing for ARM.

```bash
kubectl apply -f https://raw.githubusercontent.com/flux-framework/flux-operator/refs/heads/main/examples/dist/flux-operator-arm.yaml
```

If you haven't, make sure you CD to the chapter 4 tutorial directory.

```bash
cd /home/ubuntu/tutorial/module2
```

Create the Flux MiniCluster, which is the Custom Resource Definition (CRD) that the Flux Operator manages. This will take about 2 minutes, 20 seconds to go from creation to running, and most of that time is to pull the image.

```bash
kubectl apply -f ./manifests/flux-minicluster-lammps.yaml
```

You can use `kubectl get pods` to watch the pods transition from `Init:0/1` (this is where we add Flux to the application container and configure the cluster) to `PodInitializing` (this is when your application container is being pulled) to `Running`. This entire sequence for this image takes approximately 3 minutes. Try using `--watch` to easily monitor for updates.

```bash
kubectl get pods --watch
```
When you see `Running` you can press Control+C.

Here is a way to use `kubectl get pods` with jq to programmatically get the lead broker pod identifier, which we can use to stream logs and watch LAMMPS output.

```bash
lead_broker=$(kubectl get pods -o json | jq -r .items[0].metadata.name)
kubectl logs $lead_broker -f
```

And when you are done, clean up the MiniCluster.

```bash
kubectl delete -f ./manifests/flux-minicluster-lammps.yaml
```

<div class="alert alert-block" style="background-color:#D9A441;color:#06293D">
<span style="font-weight:600">Architecture note:</span> The operator manifest above is the <strong>ARM</strong> build, and the MiniCluster uses an ARM Flux view with a LAMMPS image built for AWS EFA on Graviton. These are matched to the EC2 instance. They will not run on an x86 machine.
</div>

<div style="background:#DCECF4;border-left:6px solid #D9A441;padding:12px 18px;color:#06293D"><strong>Module 2, Notebook 2 complete</strong></div>

Next: [Kubeflow Trainer](03_kubeflow_trainer.ipynb), into the same cluster.
